# 모델 B 학습과 외부 테스트

원본 최종 수정 기록: 2025-06-06T00:41:50+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.


In [ ]:
import pandas as pd

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
train_data_set = pd.read_csv('model_b_train_Z_score.csv')
test_data_set = pd.read_csv('model_b_test_Z_score.csv')

In [ ]:
selected_features = [
    "description length",
    "#posts",
    "nums/length username",
    "#followers",
    "#follows",
    "fullname words",
    # "private",
    # "nums/length fullname",
    # "external URL",
    # "name==username",
    # "profile pic"/

] #x_data_set에서 p-value를 검증을 했을 때 결과와 유의미하다고 판단된 특성들 만을 활용해서 학습

In [ ]:
df_X_selected = train_data_set[selected_features + ['fake']]

In [ ]:
df_insta_selected = test_data_set[selected_features + ['fake']]

In [ ]:
X = df_X_selected.drop(columns=["fake"])
y = df_X_selected["fake"]

In [ ]:
X_train, X_vali, y_train, y_vali = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

# 개별 모델 정의
model1 = GradientBoostingClassifier(random_state=42)
model2 = RandomForestClassifier(random_state=42)
model3 = lgb.LGBMClassifier(random_state=42)
model4 = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')


In [ ]:

# 소프트 보팅 앙상블
voting_clf = VotingClassifier(
    estimators=[
        ('gb', model1),
        ('rf', model2),
        ('lgbm', model3),
        ('xgb', model4)
    ],
    voting='soft'
)


In [ ]:

# 학습 및 평가
voting_clf.fit(X_train, y_train)
preds = voting_clf.predict(X_vali)

# 결과 출력
print("Voting Classifier Accuracy:", accuracy_score(y_vali, preds))
print(classification_report(y_vali, preds, target_names=["Real", "Fake"]))


In [ ]:
model2.fit(X_train,y_train)

In [ ]:
model2.score(X_vali,y_vali)

In [ ]:
X_test = df_insta_selected.drop(columns=["fake"])
y_test = df_insta_selected["fake"]

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

In [ ]:
# 4. 하이퍼파라미터 튜닝 (Random Forest)
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'min_samples_leaf': [1, 5]
}


In [ ]:
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3)
grid_rf.fit(X, y)
best_rf = grid_rf.best_estimator_

In [ ]:
model_gb = GradientBoostingClassifier(random_state=42)
model_lgb = lgb.LGBMClassifier(random_state=42)
model_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('gb', model_gb),
        ('rf', best_rf),
        ('lgbm', model_lgb),
        ('xgb', model_xgb)
    ],
    voting='soft',
    weights=[1, 2, 3, 3]
)

In [ ]:
cv_scores = cross_val_score(voting_clf, X, y, cv=5, scoring='accuracy')

In [ ]:
voting_clf.fit(X, y)
y_pred = voting_clf.predict(X_vali)

accuracy = accuracy_score(y_vali, y_pred)
report = classification_report(y_vali, y_pred, target_names=["Real", "Fake"])

In [ ]:
# 다시 구성: train_test_split을 사용하는 버전 (lightgbm/xgboost 제외하여 오류 방지)
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
import matplotlib.pyplot as plt

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'min_samples_leaf': [1, 5]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_

# 5. 다른 모델 정의
model_gb = GradientBoostingClassifier(random_state=42)

# 6. 앙상블 구성
voting_clf = VotingClassifier(
    estimators=[
        ('gb', model_gb),
        ('rf', best_rf)
    ],
    voting='soft',
    weights=[1, 2]
)

# 7. 학습 및 평가
voting_clf.fit(X_train, y_train)
y_pred_val = voting_clf.predict(X_vali)
val_accuracy = accuracy_score(y_vali, y_pred_val)
val_report = classification_report(y_vali, y_pred_val, target_names=["Real", "Fake"])

# 테스트 데이터에 적용
y_pred_test = voting_clf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_report = classification_report(y_test, y_pred_test, target_names=["Real", "Fake"])

# 결과 출력
{
    "Validation Accuracy": val_accuracy,
    "Validation Report": val_report,
    "Test Accuracy": test_accuracy,
    "Test Report": test_report,
    "Best RandomForest Params": grid_rf.best_params_
}
